# Kalman Trend Following

| Feature | Detail |
|---------|--------|
| Filter | 2-state Kalman (price + velocity) |
| Entry Long | EMA_fast > EMA_slow AND vel > 0 AND price > VWAP |
| Entry Short | EMA_fast < EMA_slow AND vel < 0 AND price < VWAP |
| Eval freq | 15 min |

## 1. Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import vectorbtpro as vbt
from numba import njit
from numba.core.errors import NumbaPerformanceWarning
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=NumbaPerformanceWarning)
warnings.filterwarnings(
    "ignore",
    message="Argument at index .* not found in SequenceTaker",
    module=r"vectorbtpro\\.utils\\.chunking",
)

In [ ]:
import multiprocessing
from numba import get_num_threads
from pathlib import Path

n_cores = multiprocessing.cpu_count()
print(f"Cores: {n_cores} | Numba threads: {get_num_threads()}")

def _fullscreen(fig):
    fig.update_layout(width=None, height=None, autosize=True,
        margin=dict(l=30, r=30, t=60, b=30),
        title=dict(font=dict(size=20), x=0.5, xanchor="center"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5))
    return fig

vbt.settings.set("plotting.pre_show_func", _fullscreen)
vbt.settings.returns.year_freq = pd.Timedelta(hours=24) * 252

## 2. Data

In [ ]:
_PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else Path(".").resolve()
# When run from notebooks/ dir: parent is project root
# When run from project root: "." is project root
# Fallback: search upward for data/
for _p in [Path(".").resolve(), Path(".").resolve().parent, Path(".").resolve().parent.parent]:
    if (_p / "data" / "EUR-USD.parquet").exists():
        _PROJECT_ROOT = _p
        break

def load_fx_data(path="data/EUR-USD.parquet", shift_hours=0):
    resolved = _PROJECT_ROOT / path
    data_raw = vbt.Data.from_parquet(str(resolved))
    symbol = data_raw.symbols[0]
    df = data_raw.data[symbol].set_index("date").sort_index()
    if shift_hours:
        df.index = df.index + pd.Timedelta(hours=shift_hours)
    raw = df.copy()
    raw.columns = [c.lower() for c in raw.columns]
    df.columns = [c.capitalize() for c in df.columns]
    data = vbt.Data.from_data({symbol: df}, tz_localize=False, tz_convert=False)
    return raw, data

raw, data = load_fx_data()
index_ns = vbt.dt.to_ns(raw.index)

# Quick stats
bars_per_day = raw.index.to_series().groupby(raw.index.date).count()
ann_factor = 252.0 * bars_per_day.mean()
print(f"Bars: {len(raw):,} | Range: {raw.index[0]} -> {raw.index[-1]}")
print(f"Ann factor: {ann_factor:.0f}")

In [ ]:
# Raw data inspection
print("=== OHLC Data ===")
print(f"Shape: {raw.shape}")
print(f"Columns: {raw.columns.tolist()}")
print(f"Dtypes:\n{raw.dtypes}")
print(f"\nNaN counts:\n{raw.isna().sum()}")
raw.head(10)

In [ ]:
raw.describe()

In [ ]:
# Data overview
fig = go.Figure(data=go.Scatter(x=raw.index, y=raw["close"], line=dict(color="#333", width=1)))
fig.update_layout(height=400, title="EUR/USD Close Price Overview")
fig.show()

In [ ]:
# Daily returns distribution
daily_close = raw["close"].resample("1D").last().dropna()
daily_rets = daily_close.pct_change().dropna()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Daily Returns Distribution", "QQ-like: Sorted Returns"))
fig.add_trace(go.Histogram(x=daily_rets.values, nbinsx=100, name="Daily Returns",
                           marker_color="#2196F3"), row=1, col=1)
sorted_rets = np.sort(daily_rets.values)
fig.add_trace(go.Scatter(y=sorted_rets, mode="lines", name="Sorted Returns",
                         line=dict(color="#FF5722")), row=1, col=2)
fig.update_layout(height=350, showlegend=False)
fig.show()

## 3. Numba Kernels

In [ ]:
@njit(nogil=True)
def find_day_boundaries_nb(index_ns):
    n = len(index_ns)
    start_idx = np.empty(n, dtype=np.int64); end_idx = np.empty(n, dtype=np.int64)
    if n == 0: return start_idx, end_idx, 0
    day_number = vbt.dt_nb.days_nb(ts=index_ns)
    current_day = day_number[0]; day_counter = 0; current_start = 0
    for i in range(1, n):
        if day_number[i] != current_day:
            start_idx[day_counter] = current_start; end_idx[day_counter] = i
            day_counter += 1; current_day = day_number[i]; current_start = i
    start_idx[day_counter] = current_start; end_idx[day_counter] = n; day_counter += 1
    return start_idx, end_idx, day_counter

@njit(nogil=True)
def kalman_filter_1d_nb(close, process_var, measurement_var):
    n = len(close)
    kf_price = np.full(n, np.nan); kf_velocity = np.full(n, np.nan)
    if n == 0: return kf_price, kf_velocity
    start = 0
    while start < n and np.isnan(close[start]): start += 1
    if start >= n: return kf_price, kf_velocity
    pe = close[start]; ve = 0.0; p11 = 1.0; p12 = 0.0; p22 = 1.0
    kf_price[start] = pe; kf_velocity[start] = ve
    for i in range(start+1, n):
        if np.isnan(close[i]):
            kf_price[i] = pe + ve; kf_velocity[i] = ve; continue
        pp = pe + ve; vp = ve
        p11p = p11 + 2*p12 + p22 + process_var; p12p = p12 + p22; p22p = p22 + process_var*0.01
        inn = close[i] - pp; S = p11p + measurement_var
        if abs(S) < 1e-15: kf_price[i] = pp; kf_velocity[i] = vp; continue
        k1 = p11p/S; k2 = p12p/S
        pe = pp + k1*inn; ve = vp + k2*inn
        p11 = (1-k1)*p11p; p12 = p12p - k1*p12p; p22 = p22p - k2*p12p
        kf_price[i] = pe; kf_velocity[i] = ve
    return kf_price, kf_velocity

@njit(nogil=True)
def compute_intraday_kalman_indicators_nb(index_ns, high, low, close, open_,
                                           process_var, measurement_var, ema_fast, ema_slow):
    kf_price, kf_velocity = kalman_filter_1d_nb(close, process_var, measurement_var)
    ema_fast_line = vbt.generic.nb.ewm_mean_1d_nb(kf_price, ema_fast, minp=1, adjust=True)
    ema_slow_line = vbt.generic.nb.ewm_mean_1d_nb(kf_price, ema_slow, minp=1, adjust=True)
    start_arr, end_arr, n_days = find_day_boundaries_nb(index_ns)
    group_lens = end_arr[:n_days] - start_arr[:n_days]
    vol_proxy = np.ones(len(close))  # No volume in FX data — use uniform weights
    vwap = vbt.indicators.nb.vwap_1d_nb(high, low, close, vol_proxy, group_lens)
    return kf_price, kf_velocity, ema_fast_line, ema_slow_line, vwap

@njit(nogil=True)
def intraday_kalman_signal_nb(c, close_arr, ema_fast_arr, ema_slow_arr,
                               velocity_arr, vwap_arr, index_ns_arr, eod_hour_arr, eod_minute_arr):
    ts_ns = index_ns_arr[c.i]
    cur_h = vbt.dt_nb.hour_nb(ts_ns); cur_m = vbt.dt_nb.minute_nb(ts_ns)
    eod_h = vbt.pf_nb.select_nb(c, eod_hour_arr); eod_m = vbt.pf_nb.select_nb(c, eod_minute_arr)
    if (cur_h > eod_h) or (cur_h == eod_h and cur_m >= eod_m):
        return False, vbt.pf_nb.ctx_helpers.in_long_position_nb(c), False, vbt.pf_nb.ctx_helpers.in_short_position_nb(c)
    if cur_m % 15 != 0: return False, False, False, False
    px = vbt.pf_nb.select_nb(c, close_arr); ef = vbt.pf_nb.select_nb(c, ema_fast_arr)
    es = vbt.pf_nb.select_nb(c, ema_slow_arr); vel = vbt.pf_nb.select_nb(c, velocity_arr)
    vw = vbt.pf_nb.select_nb(c, vwap_arr)
    if np.isnan(px) or np.isnan(ef) or np.isnan(es) or np.isnan(vel) or np.isnan(vw):
        return False, False, False, False
    il = vbt.pf_nb.ctx_helpers.in_long_position_nb(c)
    ish = vbt.pf_nb.ctx_helpers.in_short_position_nb(c)
    if not il and not ish:
        if ef > es and vel > 0 and px > vw: return True, False, False, False
        elif ef < es and vel < 0 and px < vw: return False, False, True, False
    elif il:
        if ef < es or vel < 0: return False, True, False, False
    elif ish and (ef > es or vel > 0): return False, False, False, True
    return False, False, False, False

## 4. Indicators

In [ ]:
KALMAN = vbt.IF(
    class_name="IntradayKalman", short_name="ikt",
    input_names=["index_ns", "high_minute", "low_minute", "close_minute", "open_minute"],
    param_names=["process_var", "measurement_var", "ema_fast", "ema_slow"],
    output_names=["kalman_price", "kalman_velocity", "ema_fast_line", "ema_slow_line", "vwap"],
).with_apply_func(compute_intraday_kalman_indicators_nb, takes_1d=True,
    process_var=0.001, measurement_var=1.0, ema_fast=100, ema_slow=500)
ind = KALMAN.run(index_ns, raw["high"], raw["low"], raw["close"], raw["open"],
    process_var=0.001, measurement_var=1.0, ema_fast=100, ema_slow=500,
    jitted_loop=True, jitted_warmup=True)

### Indicator Inspection

In [ ]:
ind_df = pd.DataFrame({
    "close": raw["close"].values, "kalman_price": ind.kalman_price.values,
    "velocity": ind.kalman_velocity.values, "ema_fast": ind.ema_fast_line.values,
    "ema_slow": ind.ema_slow_line.values, "vwap": ind.vwap.values,
}, index=raw.index)
print(f"Shape: {ind_df.shape} | NaN: {ind_df.isna().sum().to_dict()}")
print(f"\nVelocity stats: mean={ind_df['velocity'].mean():.6f}, std={ind_df['velocity'].std():.6f}")
ind_df.dropna().head(10)

## 5. Signal Visualization

In [ ]:
n = min(7200, len(raw)); sl = slice(-n, None)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3],
    subplot_titles=("Price + Kalman + EMAs + VWAP", "Kalman Velocity"))
fig.add_trace(go.Scatter(x=raw.index[sl], y=raw["close"].values[sl], name="Close",
    line=dict(color="#CCC", width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=raw.index[sl], y=ind.kalman_price.values[sl], name="Kalman",
    line=dict(color="#2196F3", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=raw.index[sl], y=ind.ema_fast_line.values[sl], name="EMA Fast",
    line=dict(color="#4CAF50", dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=raw.index[sl], y=ind.ema_slow_line.values[sl], name="EMA Slow",
    line=dict(color="#FF5722", dash="dash")), row=1, col=1)
fig.add_trace(go.Scatter(x=raw.index[sl], y=ind.vwap.values[sl], name="VWAP",
    line=dict(color="#9C27B0", dash="dot")), row=1, col=1)
fig.add_trace(go.Scatter(x=raw.index[sl], y=ind.kalman_velocity.values[sl], name="Velocity",
    fill="tozeroy", line=dict(color="#2196F3")), row=2, col=1)
fig.add_hline(y=0, line_color="gray", row=2, col=1)
fig.update_layout(height=700, title="Kalman Trend Following")
fig.show()

## 6. Backtest

In [ ]:
pf = vbt.Portfolio.from_signals(
    close=raw["close"], open=raw["open"], high=raw["high"], low=raw["low"],
    signal_func_nb=intraday_kalman_signal_nb,
    signal_args=(vbt.Rep("ca"), vbt.Rep("ef"), vbt.Rep("es"), vbt.Rep("va"),
        vbt.Rep("vw"), vbt.Rep("na"), vbt.Rep("eh"), vbt.Rep("em")),
    broadcast_named_args=dict(ca=raw["close"], ef=ind.ema_fast_line.values,
        es=ind.ema_slow_line.values, va=ind.kalman_velocity.values,
        vw=ind.vwap.values, na=index_ns, eh=21, em=0),
    slippage=0.00015, init_cash=1_000_000, freq="1min")
print(f"Trades: {pf.trades.count()}")

## 7. Results

### Orders & Trades Inspection

In [ ]:
# Orders log
print(f"=== Orders: {pf.orders.count()} ===")
orders_df = pf.orders.records_readable
print(orders_df.head(20).to_string())

In [ ]:
# Trades log
print(f"=== Trades: {pf.trades.count()} ===")
if pf.trades.count() > 0:
    trades_df = pf.trades.records_readable
    print(trades_df.head(20).to_string())
    print(f"\n=== Trade PnL Distribution ===")
    pnl = trades_df["PnL"].dropna()
    print(f"  Mean:   {pnl.mean():.4f}")
    print(f"  Median: {pnl.median():.4f}")
    print(f"  Std:    {pnl.std():.4f}")
    print(f"  Min:    {pnl.min():.4f}")
    print(f"  Max:    {pnl.max():.4f}")
    print(f"  Win%:   {(pnl > 0).mean()*100:.1f}%")
else:
    print("No trades generated.")

In [ ]:
# Position summary: time in market
print(f"=== Position Coverage ===")
pos = pf.position_mask
in_market_pct = pos.sum() / len(pos) * 100
print(f"  In market: {pos.sum():,} bars ({in_market_pct:.1f}%)")
print(f"  Flat:      {(~pos).sum():,} bars ({100-in_market_pct:.1f}%)")

### Portfolio Stats

In [ ]:
print("PORTFOLIO STATS")
print("=" * 50)
print(pf.stats().to_string())
if pf.trades.count() > 0:
    print(f"\nTRADE STATS\n{'='*50}")
    print(pf.trades.stats().to_string())

### Equity Curve & Drawdowns

In [ ]:
# Resample to daily for fast plotting (minute data has 3M+ points)
pf_daily = pf.resample("1D")
fig = pf_daily.plot(subplots=["cumulative_returns", "drawdowns", "underwater", "trade_pnl"])
fig.update_layout(height=900, title="Portfolio Summary (daily)")
fig.show()

### Trade Signals on Price

In [ ]:
n_bars = min(7200, len(raw))
sim_start = raw.index[-n_bars]
fig = pf.plot_trade_signals(plot_positions="zones", sim_start=sim_start)
fig.add_trace(go.Scatter(x=raw.index[-n_bars:], y=ind.kalman_price.values[-n_bars:],
    name="Kalman", line=dict(color="#2196F3")))
fig.add_trace(go.Scatter(x=raw.index[-n_bars:], y=ind.ema_fast_line.values[-n_bars:],
    name="EMA Fast", line=dict(color="#4CAF50", dash="dash")))
fig.add_trace(go.Scatter(x=raw.index[-n_bars:], y=ind.vwap.values[-n_bars:],
    name="VWAP", line=dict(color="#9C27B0", dash="dot")))
fig.update_layout(height=600, title="Trade Signals (last week)")
fig.show()

### Trade Analysis

In [ ]:
if pf.trades.count() > 0:
    fig = make_subplots(rows=2, cols=2,
        subplot_titles=("Trade PnL (%)", "MAE", "MFE", "Running Edge Ratio"))
    pf.trades.plot_pnl(pct_scale=True, fig=fig, add_trace_kwargs=dict(row=1, col=1))
    pf.trades.plot_mae(fig=fig, add_trace_kwargs=dict(row=1, col=2))
    pf.trades.plot_mfe(fig=fig, add_trace_kwargs=dict(row=2, col=1))
    pf.trades.plot_running_edge_ratio(fig=fig, add_trace_kwargs=dict(row=2, col=2))
    fig.update_layout(height=800, showlegend=False)
    fig.show()

### Monthly Returns Heatmap

In [ ]:
rets = pf_daily.returns
monthly = rets.resample("ME").apply(lambda x: (1 + x).prod() - 1)
df_m = pd.DataFrame({"return": monthly})
df_m["year"] = df_m.index.year; df_m["month"] = df_m.index.month
pivot = df_m.pivot_table(values="return", index="year", columns="month", aggfunc="first")
month_names = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
pivot.columns = month_names[:len(pivot.columns)]

fig = go.Figure(data=go.Heatmap(
    z=pivot.values * 100, x=pivot.columns.tolist(), y=[str(y) for y in pivot.index],
    colorscale="RdYlGn", texttemplate="%{z:.1f}%", textfont=dict(size=10), zmid=0))
fig.update_layout(title="Monthly Returns (%)", height=300 + len(pivot) * 30)
fig.show()

### Rolling Sharpe Ratio (Daily)

In [ ]:
# Use daily-resampled portfolio for fast rolling calc
daily_rets = pf_daily.returns
rolling_sharpe = daily_rets.rolling(252).apply(
    lambda x: x.mean() / x.std() * np.sqrt(252) if x.std() > 0 else 0)
fig = go.Figure()
fig.add_trace(go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe.values,
                         name="Rolling Sharpe (1Y)", line=dict(color="#2196F3")))
fig.add_hline(y=0, line_dash="solid", line_color="gray")
fig.add_hline(y=1, line_dash="dot", line_color="green", annotation_text="Sharpe=1")
fig.update_layout(height=350, title="Rolling 1-Year Sharpe Ratio (Daily)")
fig.show()

## Parameter Sweep

In [ ]:
ind_sweep = KALMAN.run(index_ns, raw["high"], raw["low"], raw["close"], raw["open"],
    process_var=0.001, measurement_var=1.0,
    ema_fast=vbt.Param([50, 100, 200]), ema_slow=vbt.Param([300, 500, 700]),
    jitted_loop=True, jitted_warmup=True, param_product=True)

In [ ]:
pf_sweep = vbt.Portfolio.from_signals(
    close=raw["close"], open=raw["open"], high=raw["high"], low=raw["low"],
    signal_func_nb=intraday_kalman_signal_nb,
    signal_args=(vbt.Rep("ca"), vbt.Rep("ef"), vbt.Rep("es"), vbt.Rep("va"),
        vbt.Rep("vw"), vbt.Rep("na"), vbt.Rep("eh"), vbt.Rep("em")),
    broadcast_named_args=dict(ca=raw["close"], ef=ind_sweep.ema_fast_line,
        es=ind_sweep.ema_slow_line, va=ind_sweep.kalman_velocity,
        vw=ind_sweep.vwap, na=index_ns, eh=21, em=0),
    slippage=0.00015, init_cash=1_000_000, freq="1min",
    jitted=dict(parallel=True), chunked="threadpool")

### Sweep Rankings

In [ ]:
# Parameter ranking table
sweep_summary = pd.DataFrame({
    "Sharpe": pf_sweep.sharpe_ratio,
    "Total Return": pf_sweep.total_return,
    "Max DD": pf_sweep.max_drawdown,
    "Trades": pf_sweep.trades.count(),
    "Win Rate": pf_sweep.trades.win_rate,
}).sort_values("Sharpe", ascending=False)

print("=== Top 10 Parameter Combos (by Sharpe) ===")
print(sweep_summary.head(10).to_string())
print(f"\n=== Bottom 5 ===")
print(sweep_summary.tail(5).to_string())

### Sweep Heatmaps

In [ ]:
fig = pf_sweep.sharpe_ratio.vbt.heatmap(x_level="ikt_ema_fast", y_level="ikt_ema_slow")
fig.update_layout(title="Sharpe Ratio Heatmap")
fig.show()

In [ ]:
fig = pf_sweep.total_return.vbt.heatmap(x_level="ikt_ema_fast", y_level="ikt_ema_slow")
fig.update_layout(title="Total Return Heatmap")
fig.show()

In [ ]:
fig = pf_sweep.max_drawdown.vbt.heatmap(x_level="ikt_ema_fast", y_level="ikt_ema_slow")
fig.update_layout(title="Max Drawdown Heatmap")
fig.show()

In [ ]:
fig = pf_sweep.trades.count().vbt.heatmap(x_level="ikt_ema_fast", y_level="ikt_ema_slow")
fig.update_layout(title="Trade Count Heatmap")
fig.show()

## Cross-Validation

*(Same `@vbt.cv_split` pattern)*